In [1]:
import re
import json
import pandas as pd

In [2]:
from snowflake.snowpark.session import Session
 
connection_params = {
    "user": "mohankrishna.samavedam@celanese.com",
    "authenticator": "externalbrowser",
    "account": "celanese-celanytics.privatelink",
    "warehouse": "reporting_wh",
    "database": "analytics_dev",
    # "database": "analytics_qa", #use for skipping the new brands updated in the Dev #for UAT F-hot-fix
    "schema": "snowpark",
    "role": "data_developer_gst"  }
snowpark_session = Session.builder.configs(connection_params).create()

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/7a3c88ff-a5f6-449d-ac6d-e8e3aa508e37/saml2?SAMLRequest=pZNbj9owEIX%2FSuQ%2B58rdIqwodLVIdGGBbat9M84ELBw79TgE%2ButrAkjbh92XPsWyz%2Fg7M8cZPpwK6R3BoNAqJXEQEQ8U15lQu5S8bh79PvHQMpUxqRWk5AxIHkZDZIUs6biye7WC3xWg9dxFCunlICWVUVQzFEgVKwCp5XQ9%2Fj6nSRBRhgjGOhy5lWQoHGtvbUnDsK7roG4F2uzCJIqiMBqETnWRfCHvEOXnjNJoq7mW95KT6%2BkDRBxG7QvCKRxheSv8KtR1BJ9RtlcR0qfNZukvF%2BsN8cb37iZaYVWAWYM5Cg6vq%2FnVADoHHCRTgOA3i7MVHIPSiCOzIIU6BKh0nUt2AK6LsrKOEbhVmEMWSr0TbnKzaUrKg8gGeNrPX%2BL502OPfSsGVtcbs26%2FTbZy2d0vfm3Hx9V28fPwhycvnHg%2F7jknl5xniBXM1CVd67aipOtHPb8Vb%2BIu7XRoHAdxu%2FNGvKlLVyhmm8p7C42PoBDcaNS51coZh8Zlj7V4v5%2FnPuvkXb%2FdHmQ%2B493Mhz60GOtE7tMLLxkm5PqOaGPEjP5vOsPw%2FV23B%2FrsMptNl1oKfvYetSmY%2FThS12%2BzIzI%2Fb6QUCibkOMsMILpopdT1xIDzkRJrKiDh6Er9908Y%2FQU%3D&RelayState=55022 to authenticate...
A browser window should have opened for you to complete the login. If you can't 

 pip install snowflake-connector-python[secure-local-storage]


### Get in scope data

In [3]:
SPT = snowpark_session.table('GST_CURATED.SPT').to_pandas()
all_grades = SPT['PRODUCT_CD'].unique().tolist()
all_grades = [i.lower() for i in all_grades]

In [4]:
all_grades_external = SPT[SPT["GRADE_INDICATOR"] == "Commercial"]['PRODUCT_CD'].unique().tolist()
all_grades_external = [i.lower() for i in all_grades_external]

### Get out of scope data

In [5]:
out_of_scope_brands = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_BRANDS').to_pandas()[["BRAND", "GRADE_INDICATOR"]].drop_duplicates()
out_of_scope_polymers = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_POLYMERS').to_pandas()["POLYMER"].unique().tolist()
out_of_scope_grades = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_GRADES').to_pandas()["GRADE"].unique().tolist()
out_of_scope_fillers = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_FILLERS').to_pandas()["FILLER"].unique().tolist()

## BRAND

In [6]:
out_of_scope_brands_normalized = []

for brand in out_of_scope_brands['BRAND']:
    #     if '®' in brand or '™' in brand:
    brand = re.sub(r'[^a-zA-Z\d\s]', ' ', brand).lower()
    brand = re.sub(r'\s+', ' ', brand).strip()
    out_of_scope_brands_normalized.append(brand)
    #     else:
#         print(brand)

out_of_scope_brands_normalized = pd.DataFrame({"BRANDS_NORMALIZED": out_of_scope_brands_normalized})
out_of_scope_brands_normalized = pd.concat([out_of_scope_brands_normalized, out_of_scope_brands.reset_index(drop=True)], axis=1)
out_of_scope_brands_normalized['GRADE_INDICATOR'] = out_of_scope_brands_normalized['GRADE_INDICATOR'].str.lower().str.strip()

In [7]:
out_of_scope_brands_commercial = out_of_scope_brands_normalized[out_of_scope_brands_normalized["GRADE_INDICATOR"] == "commercial"]["BRANDS_NORMALIZED"].to_list()
#out_of_scope_brands_internal = out_of_scope_brands_normalized[out_of_scope_brands_normalized["GRADE_INDICATOR"] == "internal"]["BRANDS_NORMALIZED"].to_list()
out_of_scope_brands_not_in_scope = out_of_scope_brands_normalized[out_of_scope_brands_normalized["GRADE_INDICATOR"] == "not in scope"]["BRANDS_NORMALIZED"].to_list()

In [8]:
print("brands_commerical:", out_of_scope_brands_commercial)
#print("brands_internal:", out_of_scope_brands_internal)
print("brands_not_in_scope:", out_of_scope_brands_not_in_scope)

brands_commerical: ['vitaldose', 'clarifoil', 'ecomid', 'pipelon', 'neolast']
brands_not_in_scope: ['sofpur', 'forflex', 'tarnoform', 'abistir', 'forprene', 'stirofor', 'kepital', 'omnicarb', 'cecopoly', 'compel', 'amcel', 'selar', 'vandar', 'pibiter', 'omnilon', 'blendfor', 'sikamid', 'celapex', 'pibifor', 'factor', 'nilamid', 'impet', 'nylfor', 'blueridge']


In [9]:
out_of_scope_grades = [i.lower() for i in out_of_scope_grades]

In [10]:
out_of_scope_grades_external = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_GRADES').to_pandas()["GRADE"].unique().tolist()

In [11]:
out_of_scope_grades_external = [i.lower() for i in out_of_scope_grades_external]

## Get synonyms data

In [12]:
# SYNONYM data from dev database i.e. "analytics_dev". Refer to "DEFINED_NAME" and "SYNONYMS".
synonym_df = snowpark_session.table('GST_CURATED.SYNONYM').to_pandas()

synonym_df.columns = [x.upper() if x.islower() else x for x in synonym_df.columns]
synonym_df = synonym_df.apply(lambda x: x.str.lower())

In [13]:
synonym_df

,TYPE,DEFINED_NAME,SYNONYMS
0,auto cert,mercedes-benz,mercedes-benz; daimler; daimler-benz; daimler ...
1,auto cert,vw group,vw group; vw; bentley
2,brand,abistir,abistir
3,brand,amcel,amcel; am
4,brand,at,at
...,...,...,...
378,ul property,relative thermal index - mechanical impact (rt...,relative thermal index - mechanical impact (rt...
379,ul property,relative thermal index - mechanical strength (...,relative thermal index - mechanical strength (...
380,ul property,rohs 2011/65/eu material,rohs 2011/65/eu material
381,ul property,surface resistivity,None


In [14]:
def get_synonyms(col):
    synonym_df2 = synonym_df[synonym_df['TYPE'] == col]
    
    synonym_df2 = synonym_df2[["DEFINED_NAME", "SYNONYMS"]].copy()
    synonym_df2 = synonym_df2.dropna().reset_index(drop=True)
    
    synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : True if ";" in x else False)].reset_index(drop=True)
    
    col_synonyms = synonym_df2.to_dict(orient='records')
    col_with_synonym = [item['DEFINED_NAME'] for item in col_synonyms]
    col_synonyms  = {item['DEFINED_NAME']: item['SYNONYMS'] for item in col_synonyms}
    for k in col_synonyms:
        if ";" in col_synonyms[k]:
            col_synonyms[k] = [x.strip() for x in  col_synonyms[k].split(';') if x!=k]

    return col_synonyms, col_with_synonym




In [15]:
brand_synonyms, brands_with_synonym = get_synonyms('brand')
brand_synonyms['ateva'] = ['at']
brands_with_synonym.append('ateva')

In [16]:
normalized_brand_list = out_of_scope_brands_normalized["BRANDS_NORMALIZED"].tolist()

synonyms_for_oos_brands = []
for oos_brand in normalized_brand_list:
    if oos_brand in brand_synonyms:
        synonyms_for_oos_brands.extend(brand_synonyms[oos_brand])

normalized_brand_list.extend(synonyms_for_oos_brands)

In [17]:
synonyms_for_oos_brands = []
for oos_brand in out_of_scope_brands_commercial:
    if oos_brand in brand_synonyms.keys():
        synonyms_for_oos_brands.extend(brand_synonyms[oos_brand])
out_of_scope_brands_commercial.extend(synonyms_for_oos_brands)

In [18]:
synonyms_for_oos_brands = []
# for oos_brand in out_of_scope_brands_internal:
#     if oos_brand in brand_synonyms.keys():
#         synonyms_for_oos_brands.extend(brand_synonyms[oos_brand])
# out_of_scope_brands_internal.extend(synonyms_for_oos_brands)

In [19]:
synonyms_for_oos_brands = []
for oos_brand in out_of_scope_brands_not_in_scope:
    if oos_brand in brand_synonyms.keys():
        synonyms_for_oos_brands.extend(brand_synonyms[oos_brand])
out_of_scope_brands_not_in_scope.extend(synonyms_for_oos_brands)

In [20]:
print("brands_commerical:", out_of_scope_brands_commercial)
#print("brands_internal:", out_of_scope_brands_internal)
print("brands_not_in_scope:", out_of_scope_brands_not_in_scope)

brands_commerical: ['vitaldose', 'clarifoil', 'ecomid', 'pipelon', 'neolast', 'pip', 'neo']
brands_not_in_scope: ['sofpur', 'forflex', 'tarnoform', 'abistir', 'forprene', 'stirofor', 'kepital', 'omnicarb', 'cecopoly', 'compel', 'amcel', 'selar', 'vandar', 'pibiter', 'omnilon', 'blendfor', 'sikamid', 'celapex', 'pibifor', 'factor', 'nilamid', 'impet', 'nylfor', 'blueridge', 'ff', 'ta', 'fp', 'oc', 'am', 'va', 'pb', 'ol', 'bf', 'cl', 'po', 'fa', 'im', 'br', 'blue ridge']


## Polymer

In [21]:
def normalize_polymers(polymers_list):
    polymers_normalized=[]
    for polymer in polymers_list:
        polymer = re.sub(r'[^a-zA-Z\d\s]', '', polymer).lower()
        polymers_normalized.append(re.sub(r'\s+', '', polymer).strip())
    return polymers_normalized
out_of_scope_polymers_normalized = normalize_polymers(out_of_scope_polymers)

In [22]:
polymer_synonyms, _ = get_synonyms('polymer')

synonyms_for_oos_polymers = []
for oos_polymer in out_of_scope_polymers_normalized:
    if oos_polymer in polymer_synonyms.keys():
        new_polymer_names=polymer_synonyms[oos_polymer]
        new_polymer_names = normalize_polymers(new_polymer_names)
        print(oos_polymer,": ", new_polymer_names)
        synonyms_for_oos_polymers.extend(new_polymer_names)
out_of_scope_polymers_normalized.extend(synonyms_for_oos_polymers)
out_of_scope_polymers_normalized = normalize_polymers(out_of_scope_polymers_normalized)
out_of_scope_polymers_normalized = list(set(out_of_scope_polymers_normalized))


ps :  ['polystyrene']
smah :  ['styrenemaleicanhydride']
peek :  ['polyetheretherketone', 'polyetheretherketone', 'polyether']
ptt :  ['polytrimethyleneterephthalate']
pa12 :  ['polyamide12', 'nylon12', 'pa12']
pvdf :  ['polyvinylidenefluoride']
tpo :  ['olefinicthermoplasticelastomer']


In [23]:
set(out_of_scope_polymers_normalized)

{'asa',
 'copolyester',
 'nylon12',
 'olefinicthermoplasticelastomer',
 'pa12',
 'peek',
 'polyamide12',
 'polyether',
 'polyetheretherketone',
 'polystyrene',
 'polytrimethyleneterephthalate',
 'polyvinylidenefluoride',
 'ps',
 'ptt',
 'pvdf',
 'smah',
 'styrenemaleicanhydride',
 'tpo'}

## Filler

In [24]:
out_of_scope_fillers_normalized = []
for filler in out_of_scope_fillers:
    filler = re.sub(r'[^a-zA-Z\d\s]', ' ', filler).lower()
    out_of_scope_fillers_normalized.append(re.sub(r'\s+', ' ', filler).strip())
    
out_of_scope_fillers_normalized= [i for i in out_of_scope_fillers_normalized if i!='glass filler']

In [25]:
filler_synonyms,_ = get_synonyms('filler')

In [26]:
synonyms_for_oos_fillers = []
for oos_filler in out_of_scope_fillers_normalized:
    if oos_filler in filler_synonyms.keys():
        
        out_of_scope_fillers_normalized.extend(filler_synonyms[oos_filler])
out_of_scope_fillers_normalized.extend(synonyms_for_oos_fillers)

In [27]:
out_of_scope_fillers_normalized

['natural organic fiber']

In [28]:
# out_of_scope_fillers_normalized.append("glass flake")
# out_of_scope_fillers_normalized.append("glass flakes")

In [29]:
out_of_scope_fillers_normalized = list(set(out_of_scope_fillers_normalized))

## Grades

In [30]:
def normalize_grade(s):
    # Remove special characters and convert to lower case
    return re.sub(r'\W+', '', s).lower()

In [31]:
def use_brand_synonym_in_grades(grades_list):
    new_grade_names = []
    for i in brands_with_synonym:
        brand_pattern = fr'^{re.escape(i)}\b'
        for grade in grades_list:
            grade = grade.lower()
            if re.match(brand_pattern, grade):
                for s in brand_synonyms[i]:
                    new_grade_names.append(re.sub(brand_pattern, s, grade))
    return new_grade_names

In [32]:
new_grades = use_brand_synonym_in_grades(all_grades)
all_grades = list(set(all_grades+new_grades))

In [33]:
new_grades_external = use_brand_synonym_in_grades(all_grades_external)
all_grades_external = list(set(all_grades_external+new_grades_external))

In [34]:
out_of_scope_new_grades = use_brand_synonym_in_grades(out_of_scope_grades)
out_of_scope_grades = list(set(out_of_scope_new_grades+out_of_scope_grades))

In [35]:
out_of_scope_new_grades_external = use_brand_synonym_in_grades(out_of_scope_grades_external)
out_of_scope_grades_external = list(set(out_of_scope_new_grades_external+out_of_scope_grades_external))

In [36]:
all_grades_normalized = [normalize_grade(s) for s in all_grades]
out_of_scope_grades_normalized = [normalize_grade(s) for s in out_of_scope_grades]

In [37]:
all_grades_external_normalized = [normalize_grade(s) for s in all_grades_external]
out_of_scope_grades_external_normalized = [normalize_grade(s) for s in out_of_scope_grades_external]

In [38]:
len(out_of_scope_grades_normalized), len(all_grades_normalized)

(11437, 7303)

In [39]:
out_of_scope_grades_normalized= list(set(out_of_scope_grades_normalized) - set(all_grades_normalized))

In [40]:
len(out_of_scope_grades_normalized)

11410

In [41]:
# len(out_of_scope_grades_external_normalized)

In [42]:
# len(out_of_scope_grades_normalized), len(out_of_scope_grades_external_normalized), len(all_grades_normalized), len(all_grades_external_normalized)

In [43]:
# len(SPT[SPT["GRADE_INDICATOR"] == "Commercial"]['PRODUCT_CD'].unique().tolist())

In [44]:
# len(SPT[SPT["GRADE_INDICATOR"] == "Internal"]['PRODUCT_CD'].unique().tolist())

In [45]:
# 12685+1685, 5839-4180, 14360-12685, 14370-1659

In [46]:
# all_grades_internal = SPT[SPT["GRADE_INDICATOR"] == "Internal"]['PRODUCT_CD'].unique().tolist()
# all_grades_internal = [grade_name.lower() for grade_name in all_grades_internal]
# new_grades_internal = use_brand_synonym_in_grades(all_grades_internal)
# all_grades_internal = list(set(all_grades_internal+new_grades_internal))
# len(all_grades_internal)

In [47]:
outOfScopeData = {"grades":list(set(out_of_scope_grades_normalized)),
                  "gradesExternal":list(set(out_of_scope_grades_external_normalized)),                  
                  "brands":list(set(out_of_scope_brands_normalized['BRANDS_NORMALIZED'])),
                  "brands_internal":[], #list(set(out_of_scope_brands_internal)),
                  "brands_commerical":list(set(out_of_scope_brands_commercial)),
                  "brands_not_in_scope":list(set(out_of_scope_brands_not_in_scope)),
                  "polymers":list(set(out_of_scope_polymers_normalized)),
                  "fillers":list(set(out_of_scope_fillers_normalized)),
                  "gradesInScope":list(set(all_grades_normalized)),
                  "gradesInScopeExternal":list(set(all_grades_external_normalized))}
outOfScopeData

{'grades': ['celanyla3wj8nc11021',
  'vemt1340',
  'zythtn53g50lrbn517',
  'pultrusiondevppgf400414er221black',
  'cspomgf3504af3001',
  'cspa6gf6003',
  'impetekx179',
  'laprene8mgsb50',
  'cxjkx1273',
  'hytdym100bkb254',
  'omnicarbpcpet325uv',
  'zytrslc1800nc010',
  'cxjkx1027',
  'factorpplgf20es111111',
  'cslftcfrtppa6gf6003340',
  'hythtr8620nc010',
  'crace2508bk503',
  'tecnoprenevkm24t1nero900',
  'sofprene389tx2375tdimororif389t330',
  'celanex2302icf15lw',
  'cnlb3gb20bk9005',
  'celanexjkx1218',
  'crastinhr5315hfsor516',
  'sofprenese600lc80a',
  'frib3gf25xv0',
  'tc3k8',
  'zytelfe310001nc010',
  'cncx20',
  'zytel70g33hs1lbk031x',
  'polifor5000v0afepnaturalexe',
  'laprene83e000968neutro',
  'productxc9021g',
  'sofprenezs82870a',
  'zyt2333gfh',
  'kepitalgc15',
  'celanex1700usfda',
  'pultrusionppgf4004cn15_copy',
  'cnlb2nj03nc1102duv',
  'fria3v2gy7045p',
  'zytelhtnfe250035bk420',
  'laprene8mgda35',
  'zytfr7280v0bk031',
  'lftrppgf305g',
  'foice504lfc',
  

In [48]:
with open(r"C:\Users\DSCMS5\OneDrive - CELANESE CORPORATION\Codebase\NLP-NER-Model-API\dependencies\outOfScopeData.json", "w") as fp:
    json.dump(outOfScopeData, fp)
    

In [49]:
# [i for i in out_of_scope_grades if i.startswith("at 12")]

## Others

In [50]:
# synonym_df[synonym_df['TYPE'] == 'others']

In [51]:
# col = 'others'
# synonym_df2 = synonym_df[synonym_df['TYPE'] == col]
    
# synonym_df2 = synonym_df2[["DEFINED_NAME", "SYNONYMS"]].copy()
# synonym_df2 = synonym_df2.dropna().reset_index(drop=True)


# col_synonyms = synonym_df2.to_dict(orient='records')
# col_with_synonym = [item['DEFINED_NAME'] for item in col_synonyms]
# col_synonyms  = {item['DEFINED_NAME']: item['SYNONYMS'] for item in col_synonyms}
# for k in polymer_synonyms:
#     if ";" in polymer_synonyms[k]:
#         col_synonyms[k] = [x.strip() for x in  col_synonyms[k].split(';') if x!=k]

In [52]:
# col_synonyms

In [53]:
# def get_ner(q):
#     url = env_data[env]['nerURL']
#     key = env_data[env]['nerAPIKey']

#     headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ key), 'azureml-model-deployment': 'blue' }

#     data = { "data": q}
#     body = str.encode(json.dumps(data))
    
#     try:
#         response = requests.post(url, headers=headers, json=data)
#         output = response.json()
#     except Exception as e:
#         print(q)
#         output = {}
    
#     return output

In [54]:
# output = get_ner("nylon bondable tpv")
# output

In [55]:
# fda_substrings = ['fda', 'food contact', 'food grade', 'food approval', 'food approved']
# for i in fda_substrings:
#     print(i)
#     if 'grade'in i:
#         break;